In [1]:
!pip install transformers datasets accelerate pandas

In [2]:
import pandas as pd
import random

data = []

qa_pairs = [
    ("What is DFA?", "DFA stands for Deterministic Finite Automaton, a computational model with a finite number of states and deterministic transitions."),
    ("Explain NFA", "NFA stands for Non-deterministic Finite Automaton, which allows multiple possible transitions for a given input symbol."),
    ("Difference between DFA and NFA", "DFA has exactly one transition per input symbol, while NFA can have multiple or zero transitions."),
    ("What is automata theory?", "Automata theory is the study of abstract machines and computational problems."),
    ("What is NLP?", "NLP stands for Natural Language Processing, a field of AI that enables computers to understand human language."),
    ("Explain tokenization", "Tokenization is the process of splitting text into smaller units such as words or sentences."),
    ("What is stemming?", "Stemming reduces words to their root form by removing suffixes."),
    ("What is lemmatization?", "Lemmatization converts words into their base or dictionary form."),
    ("Explain pushdown automata", "Pushdown automata are computational models that use a stack to recognize context-free languages."),
    ("What is a Turing machine?", "A Turing machine is a theoretical model capable of simulating any computation.")
]

templates = [
    "{}",
    "Explain {} in detail",
    "Describe {} clearly",
    "Discuss {} in computer science",
    "Explain {} with example",
    "What do you understand by {}",
    "Explain {} step by step",
    "Give details about {}",
    "Explain the concept of {}",
    "Write a short note on {}"
]

for i in range(100):
    for q, a in qa_pairs:
        question = "Question: " + random.choice(templates).format(q.lower())
        answer = a + " This concept is important in computer science."
        data.append({"input": question, "output": answer})

df = pd.DataFrame(data)
print("Dataset size:", len(df))
df.head()

Dataset size: 1000


,input,output
0,Question: what is dfa?,"DFA stands for Deterministic Finite Automaton,..."
1,Question: Explain explain nfa step by step,NFA stands for Non-deterministic Finite Automa...
2,Question: difference between dfa and nfa,DFA has exactly one transition per input symbo...
3,Question: Explain the concept of what is autom...,Automata theory is the study of abstract machi...
4,Question: Explain what is nlp? with example,"NLP stands for Natural Language Processing, a ..."


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [4]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

def preprocess(example):
    inputs = tokenizer(example["input"], max_length=64, padding="max_length", truncation=True)
    outputs = tokenizer(example["output"], max_length=128, padding="max_length", truncation=True)
    inputs["labels"] = outputs["input_ids"]
    return inputs

dataset = dataset.map(preprocess, batched=True)
dataset = dataset.remove_columns(["input", "output"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Step,Training Loss
10,10.791519
20,10.471714
30,10.071215
40,9.282985
50,8.324817
60,7.647248
70,7.238815
80,6.840398
90,6.479867
100,6.230593


TrainOutput(global_step=500, training_loss=5.935131065368652, metrics={'train_runtime': 157.1661, 'train_samples_per_second': 12.725, 'train_steps_per_second': 3.181, 'total_flos': 171189338112000.0, 'train_loss': 5.935131065368652, 'epoch': 2.0})

In [6]:
def generate_base(text):
    inputs = tokenizer("Question: " + text, return_tensors="pt")
    outputs = base_model.generate(
        **inputs,
        max_length=100,
        num_beams=5,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def generate_finetuned(text):
    inputs = tokenizer("Question: " + text, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_length=100,
        num_beams=5,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move models to device
base_model.to(device)
model.to(device)


def generate_base(text):
    inputs = tokenizer("Question: " + text, return_tensors="pt").to(device)

    outputs = base_model.generate(
        **inputs,
        max_length=100,
        num_beams=5,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def generate_finetuned(text):
    inputs = tokenizer("Question: " + text, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_length=100,
        num_beams=5,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [8]:
questions = [
    "What is DFA?",
    "Explain NLP",
    "Difference between DFA and NFA"
]

for q in questions:
    print("\n" + "="*60)
    print("QUESTION:", q)

    print("\n🔹 BEFORE (Base Model):")
    print(generate_base(q))

    print("\n🔹 AFTER (Fine-Tuned Model):")
    print(generate_finetuned(q))


QUESTION: What is DFA?

🔹 BEFORE (Base Model):
digital advertising agency

🔹 AFTER (Fine-Tuned Model):
Theory is a concept and a concept. This concept is important in computer science.

QUESTION: Explain NLP

🔹 BEFORE (Base Model):
NLP (Number of Persistent Loss of Efficacy).

🔹 AFTER (Fine-Tuned Model):
Computer science is a concept in computer science. This concept is important in science.

QUESTION: Difference between DFA and NFA

🔹 BEFORE (Base Model):
NFA is an acronym for National Football Association (NFA), a federation of American football teams based in Washington, D.C

🔹 AFTER (Fine-Tuned Model):
NFA is the concept of a system, and a concept in computer science. This concept is important in computer science.


In [9]:
print("\nConclusion:")
print("The fine-tuned model providesdomain-specific answers compared to the base model.")
print("It understands what automata and NLP concepts are related to after training.")


Conclusion:
The fine-tuned model providesdomain-specific answers compared to the base model.
It understands what automata and NLP concepts are related to after training.


In [10]:
model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")

print("Model saved successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully


In [11]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ac5211d005d64a041a3cd8fe8ecaa0c846c623df66b07f99404f005d781c0991
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [12]:
!pip install evaluate

import evaluate
import numpy as np

# Load metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Test data (same questions + expected answers)
test_data = [
    ("What is DFA?", "DFA stands for Deterministic Finite Automaton"),
    ("What is NLP?", "NLP stands for Natural Language Processing"),
    ("Difference between DFA and NFA", "DFA has one transition per input while NFA can have multiple transitions")
]

predictions = []
references = []

correct = 0

print("\n" + "="*70)
print("📊 EVALUATION RESULTS")
print("="*70)

for q, expected in test_data:
    pred = generate_finetuned(q)

    print("\nQUESTION:", q)
    print("EXPECTED:", expected)
    print("PREDICTED:", pred)

    predictions.append(pred)
    references.append([expected])

    # Simple accuracy (keyword match)
    if any(word.lower() in pred.lower() for word in expected.split()):
        correct += 1

# Accuracy
accuracy = correct / len(test_data)

# BLEU
bleu_score = bleu.compute(predictions=predictions, references=references)

# ROUGE
rouge_score = rouge.compute(predictions=predictions, references=[r[0] for r in references])

# Print results
print("\n" + "="*70)
print("📈 METRICS SUMMARY")
print("="*70)

print("Accuracy:", accuracy)
print("BLEU Score:", bleu_score["bleu"])
print("ROUGE-L Score:", rouge_score["rougeL"])

# Human evaluation note
print("\nHuman Evaluation:")
print("The fine-tuned model produces more relevant and domain-specific answers compared to the base model.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00



📊 EVALUATION RESULTS

QUESTION: What is DFA?
EXPECTED: DFA stands for Deterministic Finite Automaton
PREDICTED: DFA is the process of which a concept of a concept, a concept, a concept. This concept is important in computer science.

QUESTION: What is NLP?
EXPECTED: NLP stands for Natural Language Processing
PREDICTED: is a concept in computer science. This concept is important in computer science.

QUESTION: Difference between DFA and NFA
EXPECTED: DFA has one transition per input while NFA can have multiple transitions
PREDICTED: NFA stands to a design, a concept a, a concept. This concept is important for computer science...

📈 METRICS SUMMARY
Accuracy: 0.6666666666666666
BLEU Score: 0.0
ROUGE-L Score: 0.046798029556650245

Human Evaluation:
The fine-tuned model produces more relevant and domain-specific answers compared to the base model.
